# BIOSTAT707 Checkpoint 2

Name: Xiaoning Li
NetID: xl516

## AI-use statement

I used ChatGPT to help organize the notebook. I checked the cleaning rules against Checkpoint 1, reviewed the code and displayed results, and checked that the models used the same folds. The tools made coding and explanations easier, but I still needed to check the code and interpret the results. AI sometimes added more detail than necessary.


### Import packages and set data paths


In [ ]:
from pathlib import Path
import random
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

SEED = 707
random.seed(SEED)
np.random.seed(SEED)
DATA = Path("data")
OUT = Path("output")
OUT.mkdir(exist_ok=True)

record_files = sorted((DATA / "set-a").glob("*.txt"))
assert len(record_files) == 4000, f"Expected 4000 set-a files, found {len(record_files)}."
assert (DATA / "Outcomes-a.txt").is_file(), "Outcomes-a.txt is missing."


## 2.1 Modeling table


### Read set-a records and check the 48-hour window


In [ ]:
def load_set_a(files):
    tables = []

    for file in files:
        df = pd.read_csv(file)
        record_id = int(file.stem)

        # check that the record id matches the file name
        recorded_id = df.loc[df["Parameter"] == "RecordID", "Value"]
        assert len(recorded_id) == 1
        assert recorded_id.iloc[0] == record_id

        # move the record id from a row to a column
        df = df.loc[df["Parameter"] != "RecordID"].copy()
        df.insert(0, "RecordID", record_id)

        # convert elapsed time to minutes
        time = df["Time"].str.split(":", expand=True).astype(int)
        df["time_minutes"] = time[0] * 60 + time[1]
        tables.append(df)

    return pd.concat(tables, ignore_index=True)


long = load_set_a(record_files)
long = long[["RecordID", "Time", "time_minutes", "Parameter", "Value"]]

# check the number of admissions and the observation window
assert long["RecordID"].nunique() == 4000
assert long["time_minutes"].between(0, 48 * 60).all()

### Apply the Checkpoint 1 cleaning rules


In [ ]:
# keep the original values and replace -1 with missing in a separate column
long["Value_clean"] = long["Value"].replace(-1, np.nan)

# fixed screening limits carried forward from Checkpoint 1
bounds = {
    "Height": (50, 250),
    "Temp": (25, 45),
    "pH": (6, 8),
}

invalid = pd.Series(False, index=long.index)

for variable, (lower, upper) in bounds.items():
    invalid |= (
        long["Parameter"].eq(variable)
        & long["Value_clean"].notna()
        & ~long["Value_clean"].between(lower, upper)
    )

# treat zero weight as missing
invalid |= (
    long["Parameter"].eq("Weight")
    & long["Value_clean"].eq(0)
)

long.loc[invalid, "Value_clean"] = np.nan

# count original missing codes and newly excluded values separately
cleaning_summary = (
    long.assign(
        missing_code=long["Value"].eq(-1),
        excluded=invalid
    )
    .groupby("Parameter")[["missing_code", "excluded"]]
    .sum()
)

display(cleaning_summary.loc[cleaning_summary.sum(axis=1) > 0])

### Build one row per stay and join outcomes


In [ ]:
record_ids = pd.Index(
    [int(file.stem) for file in record_files],
    name="RecordID"
)

# keep one value per admission for the static variables
static_vars = ["Age", "Gender", "Height", "ICUType"]

static_rows = long.loc[long["Parameter"].isin(static_vars)]
assert not static_rows.duplicated(["RecordID", "Parameter"]).any()

static = static_rows.pivot(
    index="RecordID",
    columns="Parameter",
    values="Value_clean"
).reindex(record_ids)

# sort by time and keep the original order when timestamps match
series = long.loc[
    ~long["Parameter"].isin(static_vars + ["MechVent"])
].sort_values(["RecordID", "time_minutes"], kind="stable")

# summarize valid measurements over 48 hours, with first and last skipping missing values
summary = (
    series.groupby(["RecordID", "Parameter"])["Value_clean"]
    .agg(["count", "first", "last", "min", "max", "mean"])
    .unstack("Parameter")
)

summary.columns = [
    f"{variable}_{stat}" for stat, variable in summary.columns
]

wide = static.join(summary).reindex(record_ids)

# use zero for no valid measurements and leave the other summaries missing
count_cols = wide.columns[wide.columns.str.endswith("_count")]
wide[count_cols] = wide[count_cols].fillna(0).astype(int)

# summarize ventilation separately without treating no record as no ventilation
vent = (
    long.loc[long["Parameter"].eq("MechVent")]
    .groupby("RecordID")["Value_clean"]
    .agg(["count", "max"])
    .rename(columns={
        "count": "MechVent_count",
        "max": "MechVent_ever_recorded"
    })
)

wide = wide.join(vent)
wide["MechVent_count"] = wide["MechVent_count"].fillna(0).astype(int)

# check that the outcome file has exactly one row for each admission
outcomes = pd.read_csv(DATA / "Outcomes-a.txt")
assert outcomes["RecordID"].is_unique
assert set(outcomes["RecordID"]) == set(record_ids)
assert outcomes["In-hospital_death"].isin([0, 1]).all()

wide = wide.reset_index().merge(
    outcomes,
    on="RecordID",
    how="left",
    validate="one_to_one"
)
wide.columns.name = None

assert len(wide) == 4000
assert wide["RecordID"].is_unique

wide.to_csv(OUT / "cp2_modeling_table.csv", index=False)

### Check stay counts and mortality


In [ ]:
# Check one row per stay and complete outcomes.
assert wide["RecordID"].is_unique
assert wide["In-hospital_death"].notna().all()
assert wide["In-hospital_death"].isin([0, 1]).all()
assert set(wide["RecordID"]) == set(record_ids)

cohort_summary = pd.DataFrame({
    "Stays": [len(wide)],
    "Deaths": [int(wide["In-hospital_death"].sum())],
    "Death percentage": [100 * wide["In-hospital_death"].mean()],
    "Duplicated stay IDs": [int(wide["RecordID"].duplicated().sum())],
    "Missing outcomes": [int(wide["In-hospital_death"].isna().sum())],
})
display(cohort_summary.style.format({"Death percentage": "{:.2f}%"}))


## 2.2 Clinically motivated predictors

I used age, sex, height, and ICU type, treating sex and ICU type as categorical. For each of the 36 time-series variables, I used the mean, minimum, maximum, last valid value, count, and an indicator for no valid measurement.

I kept all measured variables, with no missingness threshold. I left out first-value summaries to reduce repetition. RecordID, the outcome, length of stay, survival time, Simplified Acute Physiology Score I (SAPS-I), and Sequential Organ Failure Assessment (SOFA) are not predictors. Only the listed columns are included in the predictor matrix.

### Create predictors and summarize their missingness


In [ ]:
# Use only the listed predictors.
static_predictors = ["Age", "Gender", "Height", "ICUType"]
time_variables = sorted(
    col.removesuffix("_count")
    for col in wide.columns
    if col.endswith("_count") and col != "MechVent_count"
)

predictor_rows = []
for col in static_predictors:
    predictor_rows.append({
        "Predictor": col, "Group": "Static",
        "Clinical rationale": {
            "Age": "Baseline vulnerability",
            "Gender": "Recorded demographic context",
            "Height": "Body-size context; retained from Checkpoint 1",
            "ICUType": "Admission setting and case mix",
        }[col],
    })

reasons = {
    "mean": "Overall physiological or laboratory state across available observations",
    "min": "Lowest observed value; may capture clinically relevant abnormalities",
    "max": "Highest observed value; may capture clinically relevant abnormalities",
    "last": "Most recent valid observation within the 48-hour window",
    "count": "Valid measurement frequency; may reflect monitoring intensity",
}
for variable in time_variables:
    for stat, reason in reasons.items():
        col = f"{variable}_{stat}"
        predictor_rows.append({"Predictor": col, "Group": stat,
                               "Clinical rationale": reason})
    missing_col = f"{variable}_never_measured"
    wide[missing_col] = wide[f"{variable}_count"].eq(0).astype(int)
    predictor_rows.append({
        "Predictor": missing_col, "Group": "Missingness",
        "Clinical rationale": "No valid measurement; may reflect testing decisions",
    })

wide["MechVent_never_measured"] = wide["MechVent_count"].eq(0).astype(int)
for col, reason in {
    "MechVent_ever_recorded": "Maximum recorded ventilation indicator; absence of a record remains missing",
    "MechVent_count": "Frequency of valid ventilation entries",
    "MechVent_never_measured": "No valid ventilation entry; distinguishes missing documentation",
}.items():
    predictor_rows.append({"Predictor": col, "Group": "Ventilation",
                           "Clinical rationale": reason})

feature_dictionary = pd.DataFrame(predictor_rows)
predictor_cols = feature_dictionary["Predictor"].tolist()
categorical_cols = ["Gender", "ICUType"]
numeric_cols = [col for col in predictor_cols if col not in categorical_cols]
forbidden = {"RecordID", "In-hospital_death", "Length_of_stay", "Survival", "SAPS-I", "SOFA"}
assert not forbidden.intersection(predictor_cols)

X = wide[predictor_cols].copy()
# Keep missing categories missing for later fold-specific handling.
for col, allowed in {"Gender": [0, 1], "ICUType": [1, 2, 3, 4]}.items():
    assert X[col].dropna().isin(allowed).all(), f"Unexpected category in {col}"
    X[col] = X[col].replace({value: str(value) for value in allowed})
y = wide["In-hospital_death"].astype(int).copy()
stay_ids = wide["RecordID"].copy()
assert stay_ids.is_unique

feature_dictionary["Missing percentage"] = [
    100 * X[col].isna().mean() for col in predictor_cols
]
group_summary = feature_dictionary.groupby("Group", sort=False).agg(
    Predictors=("Predictor", "size"),
).reset_index()
display(group_summary.style.hide(axis="index"))
display(feature_dictionary.style.format({"Missing percentage": "{:.1f}%"}).hide(axis="index"))

feature_dictionary.to_csv(OUT / "cp2_feature_dictionary.csv", index=False)
wide.to_csv(OUT / "cp2_modeling_table.csv", index=False)


## 2.3 Cross-validation design

I used five-fold stratified cross-validation (CV), with the same folds for all models and benchmarks. Each stay appears in one validation fold. Three-fold CV within each training fold selects model parameters using the area under the receiver operating characteristic curve (AUROC).

Imputation, scaling, and category encoding are fitted within each training split. GAM predictors were chosen on clinical grounds before evaluation. I report each outer-fold AUROC and its mean and standard deviation (SD).


### Create the shared five CV folds


In [ ]:
from sklearn.model_selection import StratifiedKFold

# Use the same folds for all models and benchmarks.
outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

outer_splits = list(outer_cv.split(X, y))
fold_id = np.zeros(len(y), dtype=int)
fold_rows = []

for fold, (train_idx, test_idx) in enumerate(outer_splits, start=1):
    assert set(stay_ids.iloc[train_idx]).isdisjoint(
        stay_ids.iloc[test_idx]
    )

    fold_id[test_idx] = fold

    fold_rows.append({
        "Fold": fold,
        "Training stays": len(train_idx),
        "Validation stays": len(test_idx),
        "Validation deaths": int(y.iloc[test_idx].sum()),
        "Validation death percentage": (
            100 * y.iloc[test_idx].mean()
        )
    })

assert (fold_id > 0).all()

fold_summary = pd.DataFrame(fold_rows)

display(
    fold_summary.style
    .format({"Validation death percentage": "{:.2f}%"})
    .hide(axis="index")
)

fold_assignments = pd.DataFrame({
    "RecordID": stay_ids,
    "In-hospital_death": y,
    "fold": fold_id
})

fold_assignments.to_csv(
    OUT / "cp2_fold_assignments.csv",
    index=False
)

Each fold has 3,200 training stays and 800 validation stays. Validation mortality ranges from 13.75% to 13.88%, close to the overall 13.85%, so stratification keeps the outcome proportions similar.


## 2.4 Model 1: Regularized logistic regression

I used ridge (L2) logistic regression because summaries of the same variable are correlated. Inner CV selected C from 0.001, 0.01, 0.1, 1, and 10; smaller C gives a stronger penalty. Numeric missing values were filled with training medians and numeric predictors were standardized. Categorical missing values were filled with training modes and categories were one-hot encoded.


### Set up ridge preprocessing and tuning


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score

numeric_preprocess = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])

categorical_preprocess = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocess = ColumnTransformer([
    ("numeric", numeric_preprocess, numeric_cols),
    ("categorical", categorical_preprocess, categorical_cols)
])

ridge_pipeline = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(
        solver="lbfgs",
        max_iter=5000
    ))
])

# Smaller C means stronger ridge regularization.
ridge_grid = {
    "model__C": [0.001, 0.01, 0.1, 1, 10]
}

### Tune and evaluate ridge within each outer fold


In [ ]:
ridge_oof = np.zeros(len(y))
ridge_rows = []

for fold, (train_idx, test_idx) in enumerate(outer_splits, start=1):
    inner_cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=SEED
    )

    search = GridSearchCV(
        ridge_pipeline,
        param_grid=ridge_grid,
        scoring="roc_auc",
        cv=inner_cv,
        n_jobs=1,
        error_score="raise"
    )

    search.fit(X.iloc[train_idx], y.iloc[train_idx])

    predictions = search.predict_proba(X.iloc[test_idx])[:, 1]
    ridge_oof[test_idx] = predictions

    ridge_rows.append({
        "Fold": fold,
        "Best C": search.best_params_["model__C"],
        "AUROC": roc_auc_score(y.iloc[test_idx], predictions)
    })

    print(f"Fold {fold} completed")

ridge_results = pd.DataFrame(ridge_rows)

display(
    ridge_results.style
    .format({"Best C": "{:.3f}", "AUROC": "{:.3f}"})
    .hide(axis="index")
)

ridge_summary = pd.DataFrame({
    "Model": ["Ridge logistic regression"],
    "Mean AUROC": [ridge_results["AUROC"].mean()],
    "SD": [ridge_results["AUROC"].std()]
})

display(
    ridge_summary.style
    .format({"Mean AUROC": "{:.3f}", "SD": "{:.3f}"})
    .hide(axis="index")
)

### Refit ridge on all of set-a


In [ ]:
ridge_search = GridSearchCV(
    ridge_pipeline,
    param_grid=ridge_grid,
    scoring="roc_auc",
    cv=outer_cv,
    n_jobs=1,
    error_score="raise"
)

ridge_search.fit(X, y)
ridge_model = ridge_search.best_estimator_

print("Selected C:", ridge_search.best_params_["model__C"])

### Plot standardized ridge coefficients


In [ ]:
import matplotlib.pyplot as plt

feature_names = (
    ridge_model.named_steps["preprocess"]
    .get_feature_names_out()
)

transformed_X = (
    ridge_model.named_steps["preprocess"]
    .transform(X)
)

coefficients = ridge_model.named_steps["model"].coef_[0]

# Put numeric and encoded categorical predictors on a common scale.
standardized_coefficients = (
    coefficients * transformed_X.std(axis=0)
)

ridge_coefficients = pd.DataFrame({
    "Predictor": feature_names,
    "Standardized coefficient": standardized_coefficients
})

ridge_coefficients["Predictor"] = (
    ridge_coefficients["Predictor"]
    .str.replace("numeric__", "", regex=False)
    .str.replace("categorical__", "", regex=False)
)

ridge_coefficients["Absolute coefficient"] = (
    ridge_coefficients["Standardized coefficient"].abs()
)

ridge_coefficients = ridge_coefficients.sort_values(
    "Absolute coefficient",
    ascending=False
)

ridge_coefficients.to_csv(
    OUT / "cp2_ridge_coefficients.csv",
    index=False
)

# Show the 20 largest coefficients for readability.
top_coefficients = ridge_coefficients.head(20).sort_values(
    "Standardized coefficient"
)

fig, ax = plt.subplots(figsize=(8, 7))

ax.barh(
    top_coefficients["Predictor"],
    top_coefficients["Standardized coefficient"]
)

ax.axvline(0, color="black", linewidth=1)
ax.set_xlabel("Log odds change per one standard deviation")
ax.set_title("Ridge: 20 largest standardized coefficients")

fig.tight_layout()
fig.savefig(
    OUT / "cp2_ridge_coefficients.png",
    dpi=150,
    bbox_inches="tight"
)
plt.show()

Ridge had a mean outer-CV AUROC of 0.842 (SD 0.019). All five folds selected C = 0.01, and CV on all of set-a selected the same value for the refitted model.

The coefficient plot shows the 20 largest absolute standardized coefficients. Coefficients are expressed per one standard deviation of each transformed predictor, including encoded categories. Older age was associated with higher predicted mortality, while a higher last Glasgow Coma Scale (GCS) value was associated with lower predicted mortality. Counts and missingness indicators also contributed.

## 2.5 Model 2: Non-linear effects

I used a logistic generalized additive model (GAM) with 12 predictors chosen on clinical grounds, rather than selecting them using all outcomes. Each predictor has 10 spline basis functions. Inner CV selected a common smoothing penalty from 0.1, 1, and 10. Missing values were filled with training medians inside each split.

| Predictors | Reason |
|---|---|
| Age | Baseline vulnerability |
| HR_mean, NIMAP_mean, Temp_mean | Heart rate, non-invasive mean arterial pressure, and temperature |
| GCS_last | Consciousness |
| Creatinine_last, BUN_last | Kidney function; BUN is blood urea nitrogen |
| Lactate_max | Poor perfusion or physiological stress |
| Platelets_min, WBC_mean | Platelets and white blood cell (WBC) count reflect blood and inflammatory abnormalities |
| HCO3_last, Glucose_last | Bicarbonate and glucose reflect acid–base and metabolic state |

These predictors cover several clinical domains while keeping the GAM smaller than the other models.


### Select GAM predictors and spline terms


In [ ]:
from pygam import LogisticGAM, s

gam_cols = [
    "Age",
    "HR_mean",
    "NIMAP_mean",
    "Temp_mean",
    "GCS_last",
    "Creatinine_last",
    "BUN_last",
    "Lactate_max",
    "Platelets_min",
    "WBC_mean",
    "HCO3_last",
    "Glucose_last"
]

X_gam = X[gam_cols].copy()

terms = s(0, n_splines=10)
for i in range(1, len(gam_cols)):
    terms += s(i, n_splines=10)

gam_lambdas = [0.1, 1, 10]

### Tune and evaluate the GAM within each outer fold


In [ ]:
gam_oof = np.zeros(len(y))
gam_rows = []

for fold, (train_idx, test_idx) in enumerate(outer_splits, start=1):
    X_train = X_gam.iloc[train_idx]
    y_train = y.iloc[train_idx]

    inner_cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=SEED
    )
    inner_splits = list(inner_cv.split(X_train, y_train))
    lambda_scores = []

    for lam in gam_lambdas:
        scores = []

        for inner_train, inner_test in inner_splits:
            imputer = SimpleImputer(strategy="median")

            train_values = imputer.fit_transform(
                X_train.iloc[inner_train]
            )
            test_values = imputer.transform(
                X_train.iloc[inner_test]
            )

            model = LogisticGAM(
                terms,
                lam=lam,
                max_iter=500
            )
            model.fit(train_values, y_train.iloc[inner_train])

            predictions = model.predict_proba(test_values)
            scores.append(
                roc_auc_score(y_train.iloc[inner_test], predictions)
            )

        lambda_scores.append(np.mean(scores))

    best_lam = gam_lambdas[np.argmax(lambda_scores)]

    # Refit using the entire outer training fold.
    imputer = SimpleImputer(strategy="median")
    train_values = imputer.fit_transform(X_train)
    test_values = imputer.transform(X_gam.iloc[test_idx])

    model = LogisticGAM(
        terms,
        lam=best_lam,
        max_iter=500
    )
    model.fit(train_values, y_train)

    predictions = model.predict_proba(test_values)
    gam_oof[test_idx] = predictions

    gam_rows.append({
        "Fold": fold,
        "Best lambda": best_lam,
        "AUROC": roc_auc_score(y.iloc[test_idx], predictions)
    })

    print(f"Fold {fold} completed")

gam_results = pd.DataFrame(gam_rows)

display(
    gam_results.style
    .format({"AUROC": "{:.3f}"})
    .hide(axis="index")
)

gam_summary = pd.DataFrame({
    "Model": ["Logistic GAM"],
    "Mean AUROC": [gam_results["AUROC"].mean()],
    "SD": [gam_results["AUROC"].std()]
})

display(
    gam_summary.style
    .format({"Mean AUROC": "{:.3f}", "SD": "{:.3f}"})
    .hide(axis="index")
)

### Refit the GAM on all of set-a


In [ ]:
full_cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=SEED
)
full_splits = list(full_cv.split(X_gam, y))
lambda_scores = []

for lam in gam_lambdas:
    scores = []

    for train_idx, test_idx in full_splits:
        imputer = SimpleImputer(strategy="median")
        train_values = imputer.fit_transform(X_gam.iloc[train_idx])
        test_values = imputer.transform(X_gam.iloc[test_idx])

        model = LogisticGAM(terms, lam=lam, max_iter=500)
        model.fit(train_values, y.iloc[train_idx])

        predictions = model.predict_proba(test_values)
        scores.append(roc_auc_score(y.iloc[test_idx], predictions))

    lambda_scores.append(np.mean(scores))

best_gam_lambda = gam_lambdas[np.argmax(lambda_scores)]

gam_imputer = SimpleImputer(strategy="median")
gam_values = gam_imputer.fit_transform(X_gam)

gam_model = LogisticGAM(
    terms,
    lam=best_gam_lambda,
    max_iter=500
)
gam_model.fit(gam_values, y)

print("Selected lambda:", best_gam_lambda)

### Plot three GAM effects with confidence bands


In [ ]:
plot_variables = ["Age", "HR_mean", "Temp_mean"]
plot_labels = ["Age (years)", "Mean heart rate (beats/min)", "Mean temperature (°C)"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, variable, label in zip(axes, plot_variables, plot_labels):
    term = gam_cols.index(variable)

    # Limit the plot to the observed 1st–99th percentiles.
    lower, upper = X_gam[variable].quantile([0.01, 0.99])
    grid = np.tile(np.median(gam_values, axis=0), (100, 1))
    grid[:, term] = np.linspace(lower, upper, 100)

    effect, interval = gam_model.partial_dependence(
        term=term,
        X=grid,
        width=0.95
    )

    ax.plot(grid[:, term], effect)
    ax.fill_between(
        grid[:, term],
        interval[:, 0],
        interval[:, 1],
        alpha=0.2
    )
    ax.set_xlabel(label)
    ax.set_ylabel("Partial effect on log odds")

fig.tight_layout()
fig.savefig(
    OUT / "cp2_gam_effects.png",
    dpi=150,
    bbox_inches="tight"
)
plt.show()

The GAM had a mean outer-CV AUROC of 0.824 (SD 0.021), below ridge. CV on all of set-a selected lambda = 10.

The three plots show partial contributions to log odds with approximate 95% pointwise confidence bands. They show the 1st–99th percentiles for display only; extreme observations were not removed from training.

The age curve increased and became steeper at older ages, suggesting a non-linear association consistent with greater vulnerability in older patients. 

The heart-rate curve was flat at lower values and rose above, which is consistent with physiological stress at higher heart rates. 

The temperature curve decreased toward about 37.5°C and then flattened. This suggests a non-linear association consistent with low temperature marking more severe illness, but it does not show a clear increase at high temperatures.


## 2.6 Model 3: Tree ensemble

I used a random forest with 300 trees. Inner CV compared max_features = sqrt or 0.5 and minimum leaf sizes of 5 or 15, giving four combinations. Tree depth was unrestricted. There was no early stopping. Numeric missing values were filled with training medians; categorical missing values were filled with training modes and one-hot encoded.

### Tune and evaluate the random forest


In [ ]:
from sklearn.ensemble import RandomForestClassifier

tree_preprocess = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), numeric_cols),
    ("categorical", categorical_preprocess, categorical_cols)
])

forest_pipeline = Pipeline([
    ("preprocess", tree_preprocess),
    ("model", RandomForestClassifier(
        n_estimators=300,
        random_state=SEED,
        n_jobs=1
    ))
])

forest_grid = {
    "model__max_features": ["sqrt", 0.5],
    "model__min_samples_leaf": [5, 15]
}

forest_oof = np.zeros(len(y))
forest_rows = []

for fold, (train_idx, test_idx) in enumerate(outer_splits, start=1):
    inner_cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=SEED
    )

    search = GridSearchCV(
        forest_pipeline,
        param_grid=forest_grid,
        scoring="roc_auc",
        cv=inner_cv,
        n_jobs=1,
        error_score="raise"
    )

    search.fit(X.iloc[train_idx], y.iloc[train_idx])

    predictions = search.predict_proba(X.iloc[test_idx])[:, 1]
    forest_oof[test_idx] = predictions

    forest_rows.append({
        "Fold": fold,
        "Max features": search.best_params_["model__max_features"],
        "Min samples leaf": search.best_params_["model__min_samples_leaf"],
        "AUROC": roc_auc_score(y.iloc[test_idx], predictions)
    })

    print(f"Fold {fold} completed")

forest_results = pd.DataFrame(forest_rows)

display(
    forest_results.style
    .format({"AUROC": "{:.3f}"})
    .hide(axis="index")
)

forest_summary = pd.DataFrame({
    "Model": ["Random forest"],
    "Mean AUROC": [forest_results["AUROC"].mean()],
    "SD": [forest_results["AUROC"].std()]
})

display(
    forest_summary.style
    .format({"Mean AUROC": "{:.3f}", "SD": "{:.3f}"})
    .hide(axis="index")
)

Random forest had a mean outer-CV AUROC of 0.847 (SD 0.022). The chosen settings varied across folds. Its mean AUROC was only about 0.005 above ridge, which is small relative to the observed fold-to-fold variation.


## 2.7 Comparison and final model

I evaluated SAPS-I and SOFA alone as risk scores on the same outer folds. Missing scores, including -1 codes, were excluded rather than imputed. The first comparison uses all stays for fitted models and available scores for each benchmark. I also compare all five methods on stays with both benchmark scores available.

The receiver operating characteristic (ROC) plot uses out-of-fold (OOF) predictions for fitted models and raw benchmark scores.


### Evaluate benchmarks and save model comparisons


In [ ]:
benchmark_scores = wide[["SAPS-I", "SOFA"]].replace(-1, np.nan)

benchmark_rows = []

for fold, (train_idx, test_idx) in enumerate(outer_splits, start=1):
    for score in ["SAPS-I", "SOFA"]:
        values = benchmark_scores[score].iloc[test_idx]
        observed = values.notna()

        benchmark_rows.append({
            "Model": score,
            "Fold": fold,
            "Evaluated stays": int(observed.sum()),
            "AUROC": roc_auc_score(
                y.iloc[test_idx][observed],
                values[observed]
            )
        })

benchmark_results = pd.DataFrame(benchmark_rows)

display(
    benchmark_results.style
    .format({"AUROC": "{:.3f}"})
    .hide(axis="index")
)

comparison = pd.DataFrame({
    "Ridge": ridge_results["AUROC"].to_numpy(),
    "GAM": gam_results["AUROC"].to_numpy(),
    "Random forest": forest_results["AUROC"].to_numpy(),
    "SAPS-I": benchmark_results.loc[
        benchmark_results["Model"].eq("SAPS-I"), "AUROC"
    ].to_numpy(),
    "SOFA": benchmark_results.loc[
        benchmark_results["Model"].eq("SOFA"), "AUROC"
    ].to_numpy()
}, index=[f"Fold {i}" for i in range(1, 6)]).T

comparison["Mean"] = comparison.mean(axis=1)
comparison["SD"] = comparison.iloc[:, :5].std(axis=1)

display(comparison.style.format("{:.3f}"))

comparison.to_csv(OUT / "cp2_model_comparison.csv")

oof_predictions = pd.DataFrame({
    "RecordID": stay_ids,
    "In-hospital_death": y,
    "fold": fold_id,
    "Ridge": ridge_oof,
    "GAM": gam_oof,
    "Random forest": forest_oof,
    "SAPS-I": benchmark_scores["SAPS-I"],
    "SOFA": benchmark_scores["SOFA"]
})

oof_predictions.to_csv(
    OUT / "cp2_oof_predictions.csv",
    index=False
)

### Plot ROC curves for all five methods


In [ ]:
from sklearn.metrics import roc_curve

fig, ax = plt.subplots(figsize=(7, 6))

for name in ["Ridge", "GAM", "Random forest", "SAPS-I", "SOFA"]:
    values = oof_predictions[name]
    observed = values.notna()

    fpr, tpr, thresholds = roc_curve(
        y[observed],
        values[observed]
    )
    auc = roc_auc_score(y[observed], values[observed])

    ax.plot(fpr, tpr, label=f"{name} (AUROC = {auc:.3f})")

ax.plot([0, 1], [0, 1], "--", color="gray")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("ROC curves on set-a")
ax.legend(loc="lower right")

fig.tight_layout()
fig.savefig(
    OUT / "cp2_roc_curves.png",
    dpi=150,
    bbox_inches="tight"
)
plt.show()

### Compare all methods on stays with both benchmark scores


In [ ]:
# Compare all five methods on the same stays.
complete_scores = (
    oof_predictions[["SAPS-I", "SOFA"]].notna().all(axis=1)
)

methods = ["Ridge", "GAM", "Random forest", "SAPS-I", "SOFA"]
common_rows = []

for fold in range(1, 6):
    selected = complete_scores & oof_predictions["fold"].eq(fold)
    outcomes = oof_predictions.loc[selected, "In-hospital_death"]

    row = {
        "Fold": fold,
        "Stays": int(selected.sum()),
        "Deaths": int(outcomes.sum())
    }

    for name in methods:
        row[name] = roc_auc_score(
            outcomes,
            oof_predictions.loc[selected, name]
        )

    common_rows.append(row)

common_results = pd.DataFrame(common_rows)

display(
    common_results.style
    .format({name: "{:.3f}" for name in methods})
    .hide(axis="index")
)

common_comparison = common_results.set_index("Fold")[methods].T
common_comparison.columns = [
    f"Fold {fold}" for fold in range(1, 6)
]
common_comparison["Mean"] = common_comparison.mean(axis=1)
common_comparison["SD"] = common_comparison.iloc[:, :5].std(axis=1)

display(common_comparison.style.format("{:.3f}"))

common_comparison.to_csv(
    OUT / "cp2_common_stays_comparison.csv"
)

## Conclusions and limitations


Random forest had the highest mean outer-CV AUROC, 0.847 (SD 0.022), followed by ridge, 0.842 (SD 0.019), and GAM, 0.824 (SD 0.021). SAPS-I and SOFA had mean AUROCs of 0.649 and 0.623. The ROC plot also shows similar performance for ridge and random forest, above the benchmarks.

SAPS-I was available for 3,810 stays and SOFA for 3,856. Both scores were available for 3,733 stays, including 520 deaths. In that common group, mean AUROCs were 0.843 for ridge, 0.827 for GAM, 0.847 for random forest, 0.650 for SAPS-I, and 0.620 for SOFA. The common-group table supports the same conclusion. The model–benchmark gaps were larger than the observed fold-to-fold variation, while the random forest–ridge gap was small. This is a descriptive comparison, not a formal significance test.

I selected ridge because its performance was close to random forest and its coefficients were easier to interpret. The final ridge model was refitted on all of set-a with C = 0.01. The saved pipeline includes imputation, scaling, and encoding; performance is reported from outer CV, not training predictions.

These results come from one data source and a 48-hour observation window. Several variables have substantial missingness, and monitoring features may reflect clinical decisions. The models therefore describe associations rather than causes.

### Save the final model and OOF predictions


In [ ]:
import joblib

joblib.dump(
    ridge_model,
    OUT / "cp2_final_model.joblib"
)

pd.DataFrame({
    "Predictor": predictor_cols
}).to_csv(
    OUT / "cp2_predictor_columns.csv",
    index=False
)

oof_predictions.to_csv(
    OUT / "cp2_oof_predictions.csv",
    index=False
)